In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import torch
import joblib
import gc
from sklearn.metrics import accuracy_score, average_precision_score
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.decomposition import PCA
from aeon.datasets import load_classification
from aeon.datasets.tsc_datasets import multivariate as uea_names
from datetime import datetime

# Add parent directory to path for imports
%cd ..
try:
    from distances import TimeSeriesDistance
    print("Successfully imported TimeSeriesDistance from distances.py")
except ImportError as e:
    print(f"Could not import distances.py: {e}")
%cd test_knn_multivariate
print(f"Current working directory: {os.getcwd()}")

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

DATA_PATH_UEA = "../consolidated_datasets/aeon_datasets"
RESULTS_FILE = "multivariate_knn_results.csv"
ERROR_LOG_FILE = "multivariate_failed_log.txt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# LOCAL_DATASETS = ["Weizmann", "SpokenArabicDigits"]
LOCAL_DATASETS = ["Weizmann"]

UEA_DATASETS = ["BasicMotions", "CharacterTrajectories"]
DATASETS_TO_RUN = LOCAL_DATASETS + UEA_DATASETS

# Add any UEA dataset name here (see uea_names for the full list)
# UEA_DATASETS = ["BasicMotions", "CharacterTrajectories", "Heartbeat"]

KNN_K_VALUES = [1, 3, 5, 7, 9, 11, 13, 15]

DISTANCES_TO_RUN = [
    "tmw",
    "tcot",
    "taot",
    "pow"
    "opw",
    "gow",
    "dtw",
    "awswd",
]

DISTANCE_HYPERPARAMS = {
    "opw": {
        "lambda1": 50.0,
        "lambda2": 0.1,
        "sigma": 1.0,
        "num_iter": 100
    },
    "taot": {
        "reg_lambda": 10.0,
        "time_weight": 10.0,
        "num_iter": 1000
    },
    "tcot": {
        "reg_lambda": 10.0,
        "num_iter": 1000
    },
    "awswd": {
        "reg_lambda": 10.0,
        "l_window": 5,
        "k_steep": 0.1,
        "num_sinkhorn": 100,
        "num_outer": 5
    },
    "tmw": {
        "cost_function": "L2",
        "mask_type": 2,
        "reg": 0.01,
        "max_iterations": 500,
        "thres": 1e-5,
        "eps_threshold": 0.2,
        "masked": True,
        "rescale": True
    },
    "dtw": {
        "global_constraint": None,
        "sakoe_chiba_radius": None,
        "itakura_max_slope": None
    },
    "gow": {
        "lambda1": 5.0,
        "lambda2": 0.1,
        "max_iter": 20,
        "sinkhorn_iter": 10,
        "fw_iter": 10
    },
    "pow": {
        "order_reg": 1.0,
        "sinkhorn_reg": 0.05,
        "m_mass": 0.8,
        "num_iter": 200
    }
}

print(f"Device: {DEVICE}")
print(f"Datasets to run: {DATASETS_TO_RUN}")
print(f"Distances to run: {DISTANCES_TO_RUN}")
print(f"KNN k values: {KNN_K_VALUES}")

In [ ]:
# ============================================================================
# DATASET LOADERS
# ============================================================================

def to_time_major(X):
    """Normalize aeon arrays to shape (n_samples, series_length, n_dims)."""
    if X.ndim == 2:
        return X[:, :, np.newaxis]
    if X.ndim == 3:
        return np.transpose(X, (0, 2, 1))
    raise ValueError(f"Unsupported input shape: {X.shape}")


def as_float_list(X):
    """Convert array input to a list of float32 arrays with NaNs removed."""
    return [np.nan_to_num(np.asarray(x, dtype=np.float32)) for x in X]


def load_uea_dataset(dataset_name):
    """Load any UEA multivariate dataset via aeon."""
    X_train, y_train = load_classification(
        dataset_name, split="train", extract_path=DATA_PATH_UEA
    )
    X_test, y_test = load_classification(
        dataset_name, split="test", extract_path=DATA_PATH_UEA
    )
    X_train = to_time_major(X_train)
    X_test = to_time_major(X_test)
    return as_float_list(X_train), y_train, as_float_list(X_test), y_test


def load_weizmann():
    """Load Weizmann dataset (binary masks + PCA)."""
    print("Loading Weizmann...")
    folder_path = "../consolidated_datasets/wei_dataset_feature/binary"
    file_list = os.listdir(folder_path)

    data = []
    labels = []
    subjects = []

    for file_name in file_list:
        if not file_name.endswith(".pkl"):
            continue
        parts = file_name.split("_")
        subject_name = parts[0]
        action_name = parts[1]

        file_path = os.path.join(folder_path, file_name)
        with open(file_path, "rb") as f:
            x_sample = joblib.load(f)
            if x_sample.ndim > 2:
                x_sample = x_sample.reshape(x_sample.shape[0], -1)
            data.append(x_sample)
            labels.append(action_name)
            subjects.append(subject_name)

    unique_subjects = sorted(list(set(subjects)))
    train_subjects = unique_subjects[:5]
    test_subjects = unique_subjects[5:]
    print(f"Train subjects: {train_subjects}")
    print(f"Test subjects: {test_subjects}")

    X_train_raw, y_train_raw = [], []
    X_test_raw, y_test_raw = [], []

    for i in range(len(data)):
        if subjects[i] in train_subjects:
            X_train_raw.append(data[i])
            y_train_raw.append(labels[i])
        else:
            X_test_raw.append(data[i])
            y_test_raw.append(labels[i])

    all_train_frames = np.vstack(X_train_raw)
    print(f"Fitting PCA on {all_train_frames.shape}...")
    pca = PCA(n_components=123)
    pca.fit(all_train_frames)
    print(f"Explained variance: {np.sum(pca.explained_variance_ratio_):.4f}")

    X_train = [pca.transform(x) for x in X_train_raw]
    X_test = [pca.transform(x) for x in X_test_raw]

    return as_float_list(X_train), y_train_raw, as_float_list(X_test), y_test_raw


def load_spoken_arabic():
    """Load Spoken Arabic Digits dataset."""
    print("Loading Spoken Arabic Digits...")

    def parse_file_blocks(filepath):
        data = []
        current_block = []
        with open(filepath, "r") as f:
            lines = f.readlines()
        for line in lines:
            line = line.strip()
            if not line:
                if current_block:
                    data.append(np.array(current_block, dtype=float))
                    current_block = []
            else:
                current_block.append([float(x) for x in line.split()])
        if current_block:
            data.append(np.array(current_block, dtype=float))
        return data

    train_path = "../consolidated_datasets/spoken_arabic/Train_Arabic_Digit.txt"
    test_path = "../consolidated_datasets/spoken_arabic/Test_Arabic_Digit.txt"

    train_blocks = parse_file_blocks(train_path)
    test_blocks = parse_file_blocks(test_path)

    X_train = train_blocks
    y_train = []
    for i in range(10):
        y_train.extend([i] * 660)

    X_test = test_blocks
    y_test = []
    for i in range(10):
        y_test.extend([i] * 220)

    return as_float_list(X_train), np.array(y_train), as_float_list(X_test), np.array(y_test)


LOCAL_LOADERS = {
    "Weizmann": load_weizmann,
    "SpokenArabicDigits": load_spoken_arabic
}


def load_dataset(dataset_name):
    if dataset_name in LOCAL_LOADERS:
        return LOCAL_LOADERS[dataset_name]()
    return load_uea_dataset(dataset_name)

In [ ]:
# ============================================================================
# BATCHING UTILITIES
# ============================================================================

def get_optimal_batch_size(series_length, vram_fraction=0.8):
    """Estimate optimal batch size based on available VRAM and series length."""
    if str(DEVICE) == "cpu":
        return 64
    try:
        free_mem, _ = torch.cuda.mem_get_info()
        usable_mem = free_mem * vram_fraction
        est_mem = (series_length ** 2 * 4) * 5
        est_mem = max(est_mem, 1)
        batch_size = int(usable_mem / est_mem)
        return max(1, min(batch_size, 1024))
    except Exception:
        return 32


def build_train_batches(X_train):
    """Group training samples by sequence length and build CPU batches."""
    train_indices_by_len = {}
    for idx, sample in enumerate(X_train):
        length = sample.shape[0]
        train_indices_by_len.setdefault(length, []).append(idx)

    train_batches = []
    for length, indices in train_indices_by_len.items():
        batch_size = get_optimal_batch_size(length)
        for i in range(0, len(indices), batch_size):
            batch_idxs = indices[i : i + batch_size]
            batch_samples = [X_train[idx] for idx in batch_idxs]
            batch_tensor = torch.tensor(np.stack(batch_samples), dtype=torch.float32)
            train_batches.append({
                "indices": batch_idxs,
                "tensor": batch_tensor,
                "length": length
            })
    return train_batches


def compute_distance_matrix(X_train, X_test, distance_func):
    """Precompute full distance matrix between test and train samples."""
    n_train = len(X_train)
    n_test = len(X_test)

    train_batches = build_train_batches(X_train)
    dist_matrix = np.zeros((n_test, n_train), dtype=np.float32)

    for i in range(n_test):
        test_sample = X_test[i]
        test_len = test_sample.shape[0]
        test_tensor = torch.tensor(test_sample, dtype=torch.float32).to(DEVICE).unsqueeze(0)

        for batch in train_batches:
            batch_idxs = batch["indices"]
            train_tensor = batch["tensor"].to(DEVICE)
            train_len = batch["length"]

            current_bs = len(batch_idxs)
            safe_bs = get_optimal_batch_size(max(test_len, train_len))

            if current_bs <= safe_bs:
                test_repeated = test_tensor.repeat(current_bs, 1, 1)
                with torch.no_grad():
                    dists = distance_func(test_repeated, train_tensor)
                dist_matrix[i, batch_idxs] = dists.cpu().numpy()
            else:
                for k in range(0, current_bs, safe_bs):
                    end_k = min(k + safe_bs, current_bs)
                    sub_batch_idxs = batch_idxs[k:end_k]
                    sub_train_tensor = train_tensor[k:end_k]
                    sub_bs = end_k - k
                    sub_test_repeated = test_tensor.repeat(sub_bs, 1, 1)
                    with torch.no_grad():
                        dists = distance_func(sub_test_repeated, sub_train_tensor)
                    dist_matrix[i, sub_batch_idxs] = dists.cpu().numpy()

        if (i + 1) % 10 == 0:
            print(f"Computing distances: {i + 1}/{n_test}", end="\r")

    print("")
    return dist_matrix

In [ ]:
# ============================================================================
# KNN ENGINE
# ============================================================================

def run_knn_from_distance_matrix(dist_matrix, y_train, y_test, k_values):
    """Run k-NN classification for multiple k values using precomputed distances."""
    n_test = dist_matrix.shape[0]
    n_train = dist_matrix.shape[1]
    effective_max_k = min(max(k_values), n_train)

    top_k_indices = np.argpartition(dist_matrix, effective_max_k - 1, axis=1)[:, :effective_max_k]
    row_indices = np.arange(n_test)[:, np.newaxis]
    top_k_dists = dist_matrix[row_indices, top_k_indices]
    sorted_order = np.argsort(top_k_dists, axis=1)
    top_k_indices_sorted = np.take_along_axis(top_k_indices, sorted_order, axis=1)

    all_classes = np.unique(np.concatenate([y_train, y_test]))
    n_classes = len(all_classes)
    y_test_bin = label_binarize(y_test, classes=all_classes)
    if n_classes == 2:
        y_test_bin = np.hstack([1 - y_test_bin, y_test_bin])

    results = {}
    for k in k_values:
        if k > n_train:
            continue
        predictions = []
        y_score = np.zeros((n_test, n_classes), dtype=np.float64)

        for i in range(n_test):
            k_nearest_indices = top_k_indices_sorted[i, :k]
            k_nearest_labels = y_train[k_nearest_indices]
            label_counts = {}
            for label in k_nearest_labels:
                label_counts[label] = label_counts.get(label, 0) + 1
            most_common = max(label_counts.items(), key=lambda item: item[1])[0]
            predictions.append(most_common)
            for cls_idx, cls in enumerate(all_classes):
                y_score[i, cls_idx] = label_counts.get(cls, 0) / k

        acc = accuracy_score(y_test, predictions)

        try:
            ap_per_class = []
            for cls_idx in range(n_classes):
                if y_test_bin[:, cls_idx].sum() > 0:
                    ap = average_precision_score(y_test_bin[:, cls_idx], y_score[:, cls_idx])
                    ap_per_class.append(ap)
            map_score = np.mean(ap_per_class) if ap_per_class else 0.0
        except Exception:
            map_score = 0.0

        results[k] = {"accuracy": acc, "mAP": map_score}

    return results


def select_best_k(k_results):
    """Select best k by accuracy, then mAP."""
    return max(k_results, key=lambda k: (k_results[k]["accuracy"], k_results[k]["mAP"]))

In [ ]:
# ============================================================================
# CHECKPOINTS & LOGGING
# ============================================================================

def get_processed_entries():
    """Returns a set of (dataset, distance) tuples that have already been processed."""
    if not os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE, "w") as f:
            f.write("dataset,distance,k,accuracy,mAP,time\n")
        return set()
    try:
        df = pd.read_csv(RESULTS_FILE)
        if "dataset" not in df.columns or "distance" not in df.columns:
            return set()
        return set(zip(df["dataset"], df["distance"]))
    except Exception:
        return set()


def save_result(dataset, distance_name, k, acc, map_score, duration):
    """Save best-k result to the CSV file."""
    with open(RESULTS_FILE, "a") as f:
        f.write(f"{dataset},{distance_name},{k},{acc:.5f},{map_score:.5f},{duration:.2f}\n")


def log_failure(dataset, distance_name, error_msg):
    """Log a failure to the error log file."""
    with open(ERROR_LOG_FILE, "a") as f:
        f.write(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {dataset} ({distance_name}): {error_msg}\n")

In [ ]:
# ============================================================================
# MAIN EXECUTION - BEST K ONLY
# ============================================================================

def run_multi_dataset_benchmark():
    processed_entries = get_processed_entries()
    print(f"Starting Benchmark on {len(DATASETS_TO_RUN)} Datasets: {DATASETS_TO_RUN}")
    print(f"Distances: {DISTANCES_TO_RUN}")
    print("=" * 80)

    distance_funcs = {}
    for dist_name in DISTANCES_TO_RUN:
        params = DISTANCE_HYPERPARAMS.get(dist_name, {})
        try:
            distance_funcs[dist_name] = TimeSeriesDistance(dist_name, params, device=DEVICE)
            print(f"  Initialized {dist_name.upper()}")
        except Exception as e:
            print(f"  [!] Failed to initialize {dist_name}: {e}")
    print("=" * 80)

    for dataset_name in DATASETS_TO_RUN:
        print(f"\n--- Processing {dataset_name} ---")
        try:
            X_train, y_train_raw, X_test, y_test_raw = load_dataset(dataset_name)

            le = LabelEncoder()
            all_labels = np.concatenate([y_train_raw, y_test_raw])
            le.fit(all_labels)
            y_train = le.transform(y_train_raw)
            y_test = le.transform(y_test_raw)

            n_train = len(X_train)
            n_test = len(X_test)
            length = max(x.shape[0] for x in X_train) if X_train else 0
            print(f"    Train={n_train}, Test={n_test}, MaxLen={length}")

            for dist_name, dist_func in distance_funcs.items():
                if (dataset_name, dist_name) in processed_entries:
                    print(f"    [{dist_name.upper()}] Already processed, skipping.")
                    continue

                print(f"    [{dist_name.upper()}] Computing distances and selecting best k...")

                try:
                    start_ts = time.time()
                    dist_matrix = compute_distance_matrix(X_train, X_test, dist_func)
                    dist_compute_time = time.time() - start_ts

                    knn_start = time.time()
                    k_results = run_knn_from_distance_matrix(
                        dist_matrix, y_train, y_test, KNN_K_VALUES
                    )
                    knn_time = time.time() - knn_start

                    if not k_results:
                        print("    No valid k values for this dataset.")
                        log_failure(dataset_name, dist_name, "No valid k values")
                        continue

                    best_k = select_best_k(k_results)
                    best_acc = k_results[best_k]["accuracy"]
                    best_map = k_results[best_k]["mAP"]
                    total_time = dist_compute_time + knn_time

                    print(f"    Best: k={best_k} -> Acc={best_acc:.4f}, mAP={best_map:.4f}")
                    save_result(
                        dataset_name, dist_name, best_k, best_acc, best_map, total_time
                    )

                except Exception as e:
                    print(f"    [{dist_name.upper()}] FAILED: {e}")
                    log_failure(dataset_name, dist_name, str(e))

                torch.cuda.empty_cache()

        except Exception as e:
            print(f"    [!] Dataset Load Failed: {e}")
            for dist_name in DISTANCES_TO_RUN:
                log_failure(dataset_name, dist_name, f"Dataset Load Error: {e}")

        torch.cuda.empty_cache()
        gc.collect()

    print("\n" + "=" * 80)
    print("Processing complete!")


if __name__ == "__main__":
    run_multi_dataset_benchmark()